In [0]:
catalog='Ecommerce'

###***Order Items***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_fact_path=f"{catalog}.bronze.brz_order_items_fact"

df_slv=spark.read.table(brz_fact_path)
df_slv=df_slv.dropDuplicates(['order_id','item_seq'])
df_slv=df_slv.withColumn('quantity',when(col('quantity')=='Two',2).otherwise(col('quantity')).cast(IntegerType()))\
    .withColumn('unit_price',regexp_replace(col('unit_price'),'\\$','').cast(IntegerType()))\
    .withColumn('discount_pct',regexp_replace(col('discount_pct'),'%','').cast(IntegerType()))\
    .withColumn('channel',when(col('channel')=='app','Application').when(col('channel')=='web','Website').otherwise(col('channel')))\
    .withColumn("coupon_code", lower(trim(col("coupon_code"))))\
    .withColumnRenamed('dt','date')
    
df_slv.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.silver.slv_order_items_fact')

###***Order Returns***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_fact_path=f"{catalog}.bronze.brz_order_returns_fact"

df_slv=spark.read.table(brz_fact_path)
df_slv.write.format('delta').mode('Overwrite').option('mergeSchema',True).saveAsTable(f'{catalog}.silver.slv_order_returns_fact')


###***Order Shipments***

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *

brz_fact_path=f"{catalog}.bronze.brz_order_shipments_fact"

df_slv=spark.read.table(brz_fact_path)
df_slv.write.format('delta').mode('Overwrite').option('mergeSchema',True).saveAsTable(f"{catalog}.silver.slv_order_shipments_fact")
